# Belief Path Tracking
Configuration: `config`. Figures remain local; datasets are under `experiment/data/`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
belief_path_tracking_experiment_dir = ROOT / 'experiment/research/failure_regions'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

belief_path_tracking_data_paths = ExperimentPaths(ROOT, belief_path_tracking_experiment_dir / 'config/paths.json')


# Belief tracking

Track the belief values during BP stage. To see if there exist some oscillations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from codes.ca_code import CA_5_2_3 as belief_path_tracking_ca_5_2_3, CA_9_2_5 as belief_path_tracking_ca_9_2_5
from noises_ca import CircuitBuilder as belief_path_tracking_CircuitBuilder, ErrorRate
from decoders import CythonBPOSD
from utils.graph_struct import build_tanner_graph_from_dem
from utils.graph_vis import visualize_det_on_tanner_graph, animate_hard_decisions_on_graph

belief_path_tracking_data_dir = belief_path_tracking_data_dir = belief_path_tracking_data_paths.output('circuit_level')
BELIEF_DIR = belief_path_tracking_data_paths.output('belief_track')
%load_ext autoreload
%autoreload 2


### Load structural cases

Read the named structural-feature input and prepare decoder contexts for the selected code and p. The stored outcome labels are used to choose examples; they do not predetermine the behavior of every trace.


In [ ]:
belief_path_tracking_ca_codes = {
    'CA_5_2_3': belief_path_tracking_ca_5_2_3(),
    'CA_9_2_5': belief_path_tracking_ca_9_2_5(),
}

labels = {
    '00': 'bf',
    '11': 'bs',
    '01': 'bfts',
    '10': 'bstf',
}


In [ ]:
out_dir = belief_path_tracking_data_paths.input('trapping_structure')
belief_path_tracking_types = {
    'code': str,
    'p': float,
    'label': str,
    'detector_events': str,
    'WL_hash': str,
}

belief_path_tracking_analysis_df = pd.read_csv(belief_path_tracking_data_paths.input('event_graph_features_csv'), dtype=belief_path_tracking_types)
belief_path_tracking_analysis_df = belief_path_tracking_analysis_df[belief_path_tracking_analysis_df['code'] == 'CA_9_2_5']
belief_path_tracking_analysis_df


## Specific samples

11: both succeed -> No oscillation

01: BPOSD fails but Tess succeeds -> Oscillation

In [ ]:
row_11 = belief_path_tracking_analysis_df[belief_path_tracking_analysis_df['label'] == '11'].iloc[-1]
error_rate = float(row_11['p'])
max_iter = 60
belief_path_tracking_code = belief_path_tracking_ca_codes[row_11['code']]
belief_path_tracking_builder = belief_path_tracking_CircuitBuilder(belief_path_tracking_code)
belief_path_tracking_circ = belief_path_tracking_builder.get_circuit_level_circuit(error_rate, d_rounds=3)
compiled_decoder = CythonBPOSD(belief_path_tracking_code, error_rate, max_iter).compile_decoder_for_dem(dem=belief_path_tracking_circ.detector_error_model())


In [ ]:
row_11


In [ ]:
prediction = compiled_decoder.decoder.decode(np.asarray(list(row_11['detector_events']), dtype=int).astype(bool))
belief_history = compiled_decoder.decoder.belief_history.copy()
info = {'prediction': prediction, 'belief_history_shape': belief_history.shape}
info


In [ ]:
belief_delta = np.ptp(belief_history, axis=0)
variables_to_plot = np.argsort(belief_delta)[::-1]
belief_path_tracking_fig, ax = plt.subplots(figsize=(10, 4))

for var in variables_to_plot:
    ax.plot(belief_history[:, var], marker='o', label=f'var {var}')
ax.axhline(0, color='black', linewidth=1, alpha=0.4)
ax.set_xlabel('iteration')
ax.set_ylabel('belief / LLR')
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
belief_path_tracking_fig.tight_layout()


In [ ]:
belief_path_tracking_fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(belief_history.T, aspect='auto', origin='lower')
ax.set_xlabel('iteration')
ax.set_ylabel('variable index')
belief_path_tracking_fig.colorbar(im, ax=ax, label='belief / LLR')
belief_path_tracking_fig.tight_layout()


### Trace a second selected input

Rebuild the decoder for the next recorded row and inspect its belief history using the same visualization procedure. Compare the actual histories instead of assuming every disagreement must oscillate.


In [ ]:
row_01 = belief_path_tracking_analysis_df.iloc[-1]
error_rate = float(row_01['p'])
max_iter = 60
belief_path_tracking_code = belief_path_tracking_ca_codes[row_01['code']]
belief_path_tracking_builder = belief_path_tracking_CircuitBuilder(belief_path_tracking_code)
belief_path_tracking_circ = belief_path_tracking_builder.get_circuit_level_circuit(error_rate, d_rounds=3)
compiled_decoder = CythonBPOSD(belief_path_tracking_code, error_rate, max_iter).compile_decoder_for_dem(dem=belief_path_tracking_circ.detector_error_model())


In [ ]:
prediction = compiled_decoder.decoder.decode(np.asarray(list(row_01['detector_events']), dtype=int).astype(bool))
belief_history = compiled_decoder.decoder.belief_history.copy()
info = {'prediction': prediction, 'belief_history_shape': belief_history.shape}
info


In [ ]:
belief_delta = np.ptp(belief_history, axis=0)
variables_to_plot = np.argsort(belief_delta)[::-1]
belief_path_tracking_fig, ax = plt.subplots(figsize=(10, 4))

for var in variables_to_plot:
    ax.plot(belief_history[:, var], marker='o', label=f'var {var}')
ax.axhline(0, color='black', linewidth=1, alpha=0.4)
ax.set_xlabel('iteration')
ax.set_ylabel('belief / LLR')
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
belief_path_tracking_fig.tight_layout()


In [ ]:
belief_path_tracking_fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(belief_history.T, aspect='auto', origin='lower')
ax.set_xlabel('iteration')
ax.set_ylabel('variable index')
belief_path_tracking_fig.colorbar(im, ax=ax, label='belief / LLR')
belief_path_tracking_fig.tight_layout()


In [ ]:
np.savetxt(belief_path_tracking_data_paths.output('belief_history_01_txt'), belief_history)


### Belief value dynamics analysis

- Fixed Point：in the last few rounds, the hard decision remains unchanged, and the LLR step norm is sufficiently small.
- Oscillation：the final hard decision repeats with a fixed period, and the corresponding LLR also repeats approximately.
- Unclassified：if the above two conditions are not met, it usually means that the history is still drifting or the threshold is set too strictly.

In [ ]:
top_k = 8
fixed_window = 3
fixed_delta_tol = 1e-09
oscillation_period = 2
oscillation_delta_tol = 1e-09
belief_history = np.loadtxt(belief_path_tracking_data_paths.output('belief_history_01_txt'))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

num_iterations, num_variables = belief_history.shape
if num_iterations < 2:
    raise ValueError('Belief history must contain at least 2 iterations for analysis.')
tracked_pcm = compiled_decoder.decoder.H
tracked_syndrome = np.asarray(list(row_01['detector_events']), dtype=np.uint8) % 2
hard_history = (belief_history < 0).astype(np.uint8)
belief_step = np.diff(belief_history, axis=0)
hard_step = np.diff(hard_history, axis=0)
llr_step_l2 = np.r_[np.nan, np.linalg.norm(belief_step, axis=1)]
llr_step_linf = np.r_[np.nan, np.max(np.abs(belief_step), axis=1)]
hard_step_hamming = np.r_[np.nan, np.count_nonzero(hard_step, axis=1)]
syndrome_from_hard_history = hard_history @ tracked_pcm.T % 2
syndrome_residual_history = syndrome_from_hard_history ^ tracked_syndrome[None, :]
syndrome_residual_weight = syndrome_residual_history.sum(axis=1)
sign_flip_count = np.count_nonzero(np.diff(hard_history, axis=0), axis=0)
llr_range = np.ptp(belief_history, axis=0)
final_abs_llr = np.abs(belief_history[-1])
final_hard_decision = hard_history[-1]

def _tail_max_step_l2(history, window) -> float:
    """the maximum L2 norm of stepwise changes in the tail of the history, over the specified window size"""

    if history.shape[0] < window:
        return np.nan
    tail_step = np.diff(history[-window:], axis=0)

    if tail_step.shape[0] == 0:
        return 0.0
    return float(np.max(np.linalg.norm(tail_step, axis=1)))

def _is_fixed_point(
    history,
    hard,
    window,
    delta_tol,
) -> bool:
    """check if the belief path has converged to a fixed point in the last `window` iterations, and change of beliefs is within a tolerance of `delta_tol`"""

    if history.shape[0] < window:
        return False
    tail_hard = hard[-window:]
    hard_is_fixed = np.all(tail_hard == tail_hard[-1])

    return bool(hard_is_fixed and _tail_max_step_l2(history, window) <= delta_tol)

def _oscillation_repeat_delta(history, period) -> float:
    """the maximum L2 norm of the difference between the last two periods of the history"""

    if history.shape[0] < 2 * period:
        return np.nan
    previous = history[-2 * period:-period]
    current = history[-period:]

    return float(np.max(np.linalg.norm(current - previous, axis=1)))

def _is_periodic_oscillation(
    history,
    hard,
    period,
    delta_tol,
) -> bool:
    """check if the belief path is exhibiting a periodic oscillation with the specified period, and change between repeats is within a tolerance of `delta_tol`"""

    if history.shape[0] < 2 * period:
        return False
    previous_hard = hard[-2 * period:-period]
    current_hard = hard[-period:]
    repeats = np.array_equal(previous_hard, current_hard)
    nontrivial = any((not np.array_equal(current_hard[0], state) for state in current_hard[1:]))

    return bool(repeats and nontrivial and (_oscillation_repeat_delta(history, period) <= delta_tol))

fixed_point_detected = _is_fixed_point(belief_history, hard_history, fixed_window, fixed_delta_tol)
oscillation_detected = _is_periodic_oscillation(belief_history, hard_history, oscillation_period, oscillation_delta_tol)
if fixed_point_detected:
    belief_path_classification = f'Fixed Point(window={fixed_window})'
elif oscillation_detected:
    belief_path_classification = f'Oscillation(period={oscillation_period})'
else:
    belief_path_classification = 'Unclassified'

belief_dynamics_summary = pd.DataFrame([{
    'num_iterations': num_iterations,
    'num_variables': num_variables,
    'num_detectors': tracked_pcm.shape[0],
    'classification': belief_path_classification,
    'final_syndrome_residual_weight': int(syndrome_residual_weight[-1]),
    'initial_syndrome_residual_weight': int(syndrome_residual_weight[0]),
    'last_llr_step_l2': float(llr_step_l2[-1]) if not np.isnan(llr_step_l2[-1]) else np.nan,
    'last_llr_step_linf': float(llr_step_linf[-1]) if not np.isnan(llr_step_linf[-1]) else np.nan,
    'tail_max_llr_step_l2': _tail_max_step_l2(belief_history, fixed_window),
    'oscillation_repeat_delta': _oscillation_repeat_delta(belief_history, oscillation_period),
    'last_hard_step_hamming': float(hard_step_hamming[-1]) if not np.isnan(hard_step_hamming[-1]) else np.nan,
    'total_sign_flips': int(sign_flip_count.sum()),
}])
belief_variable_summary = pd.DataFrame({
    'variable': np.arange(num_variables),
    'llr_initial': belief_history[0],
    'llr_final': belief_history[-1],
    'abs_llr_final': final_abs_llr,
    'llr_range': llr_range,
    'sign_flip_count': sign_flip_count,
    'final_hard_decision': final_hard_decision,
}).sort_values(['sign_flip_count', 'llr_range', 'abs_llr_final'], ascending=[False, False, True]).reset_index(drop=True)


In [ ]:
belief_dynamics_summary


In [ ]:
belief_variable_summary.head(top_k)


In [ ]:
top_variables = belief_variable_summary.head(top_k)['variable'].to_numpy(dtype=int)
iteration_axis = np.arange(num_iterations)
belief_path_tracking_fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(iteration_axis, np.nan_to_num(llr_step_l2, nan=0.0), marker='o', label='LLR step L2')
axes[0, 0].plot(iteration_axis, np.nan_to_num(llr_step_linf, nan=0.0), marker='s', label='LLR step Linf')
axes[0, 0].set_xlabel('iteration')
axes[0, 0].set_ylabel('step norm')
axes[0, 0].set_title('LLR update size')
axes[0, 0].legend()

axes[0, 1].plot(iteration_axis, syndrome_residual_weight, marker='o', label='syndrome residual weight')
axes[0, 1].plot(iteration_axis, np.nan_to_num(hard_step_hamming, nan=0.0), marker='s', label='hard-decision Hamming step')
axes[0, 1].set_xlabel('iteration')
axes[0, 1].set_ylabel('count')
axes[0, 1].set_title('Hard decision and syndrome residual')
axes[0, 1].legend()

for variable in top_variables:
    axes[1, 0].plot(iteration_axis, belief_history[:, variable], marker='o', label=f'var {variable}')
axes[1, 0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1, 0].set_xlabel('iteration')
axes[1, 0].set_ylabel('belief / LLR')
axes[1, 0].set_title('Most volatile variable trajectories')
axes[1, 0].legend(ncol=2, fontsize=8)

axes[1, 1].bar(np.arange(len(top_variables)), belief_variable_summary.head(top_k)['sign_flip_count'])
axes[1, 1].set_xticks(np.arange(len(top_variables)))
axes[1, 1].set_xticklabels([str(v) for v in top_variables])
axes[1, 1].set_xlabel('variable')
axes[1, 1].set_ylabel('sign flip count')
axes[1, 1].set_title('Variable sign instability')
belief_path_tracking_fig.tight_layout()


In [ ]:
from utils.graph_struct import build_tanner_graph_from_dem
from utils.graph_vis import animate_hard_decisions_on_graph

belief_path_tracking_tg, belief_path_tracking_ = build_tanner_graph_from_dem(belief_path_tracking_circ.detector_error_model())
hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
belief_path_tracking_ani = animate_hard_decisions_on_graph(belief_path_tracking_tg, row_01['detector_events'], hard_decisions_list)

belief_path_tracking_ani.show()


In [ ]:
import plotly.io as pio

belief_path_tracking_ani.save(belief_path_tracking_data_paths.output('hard_decision_evolution_gif'), writer='pillow')


## Additional belief histories and shared-syndrome cases


### Revisit cases and shared syndromes

The next block reloads the analysis cases, traces selected rows and examines repeated detector patterns. Multiple physical errors can share the same syndrome and need not share a logical label.


In [ ]:
belief_path_tracking_types = {
    'code': str,
    'p': float,
    'detector_events': str,
    'error_events': str,
    'true_observables': str,
    'bposd_pred_observables': str,
    'tesseract_pred_observables': str,
}

belief_path_tracking_analysis_df = pd.read_csv(belief_path_tracking_data_paths.output('ca_bfts_cases_csv_1'), dtype=belief_path_tracking_types)
belief_path_tracking_analysis_df = belief_path_tracking_analysis_df[belief_path_tracking_analysis_df['code'] == 'CA_9_2_5']
belief_path_tracking_analysis_df


In [ ]:
row_01 = belief_path_tracking_analysis_df.iloc[-1]
error_rate = float(row_01['p'])
max_iter = 60
belief_path_tracking_code = belief_path_tracking_ca_codes[row_01['code']]
belief_path_tracking_builder = belief_path_tracking_CircuitBuilder(belief_path_tracking_code)
belief_path_tracking_circ = belief_path_tracking_builder.get_circuit_level_circuit(error_rate, flip_meas=0, d_rounds=1)
compiled_decoder = CythonBPOSD(belief_path_tracking_code, error_rate, max_iter).compile_decoder_for_dem(dem=belief_path_tracking_circ.detector_error_model())


In [ ]:
row_01


In [ ]:
prediction = compiled_decoder.decoder.decode(np.asarray(list(row_01['detector_events']), dtype=int).astype(bool))
belief_history = compiled_decoder.decoder.belief_history.copy()
info = {
    'prediction': prediction,
    'belief_history_shape': belief_history.shape,
    'is_bp_converged': compiled_decoder.decoder.is_bp_converged,
    'is_osd_satisfied': compiled_decoder.decoder.is_osd_satisfied,
}

info


In [ ]:
np.savetxt(belief_path_tracking_data_paths.output('parameter_1_p_parameter_2_belief_history_01_txt', parameter_1=row_01['code'], parameter_2=row_01['p']), belief_history)


In [ ]:
belief_delta = np.ptp(belief_history, axis=0)
variables_to_plot = np.argsort(belief_delta)[::-1]
belief_path_tracking_fig, ax = plt.subplots(figsize=(10, 4))

for var in variables_to_plot:
    ax.plot(belief_history[:, var], marker='o', label=f'var {var}')
ax.axhline(0, color='black', linewidth=1, alpha=0.4)
ax.set_xlabel('iteration')
ax.set_ylabel('belief / LLR')
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
belief_path_tracking_fig.tight_layout()


In [ ]:
belief_path_tracking_fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(belief_history.T, aspect='auto', origin='lower')
ax.set_xlabel('iteration')
ax.set_ylabel('variable index')
belief_path_tracking_fig.colorbar(im, ax=ax, label='belief / LLR')
belief_path_tracking_fig.tight_layout()


In [ ]:
from utils.graph_struct import build_tanner_graph_from_dem
from utils.graph_vis import animate_hard_decisions_on_graph

belief_path_tracking_tg, belief_path_tracking_ = build_tanner_graph_from_dem(belief_path_tracking_circ.detector_error_model())
hard_history = (belief_history < 0).astype(np.uint8)
hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
belief_path_tracking_ani = animate_hard_decisions_on_graph(belief_path_tracking_tg, row_01['detector_events'], hard_decisions_list, err_events=row_01['error_events'])

belief_path_tracking_ani.show()


In [ ]:
belief_path_tracking_fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(hard_history.T, aspect='auto', origin='lower')
ax.set_xlabel('iteration')
ax.set_ylabel('variable index')
belief_path_tracking_fig.colorbar(im, ax=ax, label='hard decision')
belief_path_tracking_fig.tight_layout()


### Compare explanations of the same syndrome

Group the recorded cases by detector pattern before tracing individual examples. This exposes ambiguity hidden by a syndrome-only lookup; the selected examples are not a population estimate.


In [ ]:
belief_path_tracking_analysis_df.groupby('detector_events').size().sort_values(ascending=False)[:20]


In [ ]:
belief_path_tracking_analysis_df.groupby('detector_events').agg(
    true_obs_cnt=('true_observables', 'unique'),
    bposd_pred_obs_cnt=('bposd_pred_observables', 'unique'),
    tess_pred_obs_cnt=('tesseract_pred_observables', 'unique'),
    cnt=('detector_events', 'size'),
).sort_values('cnt', ascending=False).head(20)


In [ ]:
belief_path_tracking_analysis_df[belief_path_tracking_analysis_df['detector_events'] == '11010000000010']


In [ ]:
row = belief_path_tracking_analysis_df[belief_path_tracking_analysis_df['detector_events'] == '11010000000010'].iloc[0]
error_rate = float(row['p'])
max_iter = 60
belief_path_tracking_code = belief_path_tracking_ca_codes[row['code']]
belief_path_tracking_builder = belief_path_tracking_CircuitBuilder(belief_path_tracking_code)
belief_path_tracking_circ = belief_path_tracking_builder.get_circuit_level_circuit(error_rate, flip_meas=0, d_rounds=1)
compiled_decoder = CythonBPOSD(belief_path_tracking_code, error_rate, max_iter).compile_decoder_for_dem(dem=belief_path_tracking_circ.detector_error_model())


In [ ]:
prediction = compiled_decoder.decoder.decode(np.asarray(list(row['detector_events']), dtype=int).astype(bool))
belief_history = compiled_decoder.decoder.belief_history.copy()
info = {
    'prediction': prediction,
    'belief_history_shape': belief_history.shape,
    'is_bp_converged': compiled_decoder.decoder.is_bp_converged,
    'is_osd_satisfied': compiled_decoder.decoder.is_osd_satisfied,
}

info


In [ ]:
belief_path_tracking_tg, belief_path_tracking_ = build_tanner_graph_from_dem(belief_path_tracking_circ.detector_error_model())
hard_history = (belief_history < 0).astype(np.uint8)
hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
belief_path_tracking_ani = animate_hard_decisions_on_graph(belief_path_tracking_tg, row['detector_events'], hard_decisions_list, err_events=row['error_events'])

belief_path_tracking_ani.show()


In [ ]:
row = belief_path_tracking_analysis_df[belief_path_tracking_analysis_df['detector_events'] == '00011100100000'].iloc[0]
error_rate = float(row['p'])
max_iter = 60
belief_path_tracking_code = belief_path_tracking_ca_codes[row['code']]
belief_path_tracking_builder = belief_path_tracking_CircuitBuilder(belief_path_tracking_code)
belief_path_tracking_circ = belief_path_tracking_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=error_rate, p_meas=0.0), d_rounds=1)
compiled_decoder = CythonBPOSD(belief_path_tracking_code, error_rate, max_iter).compile_decoder_for_dem(dem=belief_path_tracking_circ.detector_error_model())


In [ ]:
prediction = compiled_decoder.decoder.decode(np.asarray(list(row['detector_events']), dtype=int).astype(bool))
belief_history = compiled_decoder.decoder.belief_history.copy()
info = {
    'prediction': prediction,
    'belief_history_shape': belief_history.shape,
    'is_bp_converged': compiled_decoder.decoder.is_bp_converged,
    'is_osd_satisfied': compiled_decoder.decoder.is_osd_satisfied,
}

info


In [ ]:
belief_delta = np.ptp(belief_history, axis=0)
variables_to_plot = np.argsort(belief_delta)[::-1]
belief_path_tracking_fig, ax = plt.subplots(3, 1, figsize=(12, 12))
ax = ax.flatten()

for var in variables_to_plot:
    ax[0].plot(belief_history[:, var], marker='o', label=f'var {var}')
ax[0].axhline(0, color='black', linewidth=1, alpha=0.4)
ax[0].set_xlabel('iteration')
ax[0].set_ylabel('belief / LLR')
ax[0].legend(ncol=2, fontsize=8)
ax[0].grid(True, alpha=0.3)

im = ax[1].imshow(belief_history.T, aspect='auto', origin='lower')
ax[1].set_xlabel('iteration')
ax[1].set_ylabel('variable index')
belief_path_tracking_fig.colorbar(im, ax=ax[1], label='belief / LLR')
hard_history = (belief_history < 0).astype(np.uint8)

im = ax[2].imshow(hard_history.T, aspect='auto', origin='lower')
ax[2].set_xlabel('iteration')
ax[2].set_ylabel('variable index')
belief_path_tracking_fig.colorbar(im, ax=ax[2], label='hard decision')
belief_path_tracking_fig.tight_layout()


In [ ]:
belief_path_tracking_tg, belief_path_tracking_ = build_tanner_graph_from_dem(belief_path_tracking_circ.detector_error_model())
hard_history = (belief_history < 0).astype(np.uint8)
hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
belief_path_tracking_ani = animate_hard_decisions_on_graph(belief_path_tracking_tg, row['detector_events'], hard_decisions_list, err_events=row['error_events'])

belief_path_tracking_ani.show()


In [ ]:
row


# Graph Vis
Configuration: `config`. Figures remain local; datasets are under `experiment/data/`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
graph_vis_experiment_dir = ROOT / 'experiment/research/failure_regions'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

graph_vis_data_paths = ExperimentPaths(ROOT, graph_vis_experiment_dir / 'config/paths.json')


# Graph Visualize

In [ ]:
import multiprocessing
import sys
from pathlib import Path
from collections import defaultdict, Counter
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from codes.ca_code import CA_5_2_3 as graph_vis_ca_5_2_3, CA_9_2_5 as graph_vis_ca_9_2_5
from src.noises_ca import CircuitBuilder as graph_vis_CircuitBuilder

%load_ext autoreload
%autoreload 2


### Prepare graph representations

Construct matching/Tanner graph views and visualization helpers for the selected circuit. The graph panels show different representations of the same decoding example.


In [ ]:
graph_vis_ca_codes = {'CA_5_2_3': graph_vis_ca_5_2_3(), 'CA_9_2_5': graph_vis_ca_9_2_5()}
p_values = np.linspace(1e-08, 0.01, 10)
max_shots = 1000000000
max_errors = 2000


In [ ]:
from experiment.utils import dem_graph

extract_dem_info = dem_graph.extract_dem_info
extract_coord_from_graph = dem_graph.extract_coord_from_graph

def build_matching_graph_from_dem(dem: stim.DetectorErrorModel) -> tuple[nx.Graph, pd.DataFrame, Counter, int]:
    """
    Builds a matching graph from the Detector Error Model (DEM).

    Returns
    ----------
        G: networkx.Graph 
            nodes -> detectors and edges -> error mechanism. 
            Edge attributes:
            * multiplicity: number of shared terms 
            * weight: minimum -log(p_error) across shared terms.
        terms_df: DataFrame 
            columns ['p_error', 'n_det', 'n_obs', 'det_ids', 'obs_ids'] for each error(...) term in the DEM.
        boundary_weight: Counter 
            mapping detector_id -> count of single-detector error(...) terms it appears in (boundary detectors).
        hyperedges: int
            count of error(...) terms that involve more than 2 detectors.
    """

    terms_df, _, _ = extract_dem_info(dem)
    coords = dem.get_detector_coordinates()
    x_min, x_max, y_min, y_max = (min((c[0] for c in coords.values())), max((c[0] for c in coords.values())), min((c[1] for c in coords.values())), max((c[1] for c in coords.values())))
    x_center, y_center = ((x_min + x_max) / 2, (y_min + y_max) / 2)
    G = nx.Graph()
    detector_nodes = [(c[0], dict(coord=c[1])) for c in coords.items()]
    G.add_nodes_from(detector_nodes)
    node_cnt = G.number_of_nodes()
    G.graph['virtual_boundary_nodes'] = []
    vnode_id = 0
    boundary_weight = Counter()
    hyperedges = 0

    for _, row in terms_df.iterrows():
        det_ids = list(row['det_ids'])
        p_err = row['p_error']
        w = -np.log(max(p_err, 1e-300)) if np.isfinite(p_err) and p_err > 0 else np.nan

        if len(det_ids) == 2:
            a, b = (int(det_ids[0]), int(det_ids[1]))
            if G.has_edge(a, b):
                G[a][b]['multiplicity'] += 1
                if np.isfinite(w):
                    G[a][b]['weight'] = min(G[a][b]['weight'], w)
            else:
                G.add_edge(a, b, multiplicity=1, weight=w)
        elif len(det_ids) == 1:
            d = int(det_ids[0])
            vnode = f'B{vnode_id}'
            coord = [coords[d][0] - 1, coords[d][1] + 0.5, *coords[d][2:]] if coords[d][0] < x_center else [coords[d][0] + 1, coords[d][1] + 0.5, *coords[d][2:]]
            G.add_node(vnode, coord=coord)
            G.add_edge(vnode, d, multiplicity=1, weight=w)
            G.graph['virtual_boundary_nodes'].append(vnode)
            vnode_id += 1
            boundary_weight[d] += 1
        elif len(det_ids) > 2:
            center_coord = np.mean([coords[int(d)] for d in det_ids], axis=0).tolist()
            G.add_node(node_cnt, coord=center_coord)
            for d in det_ids:
                G.add_edge(node_cnt, int(d), multiplicity=1, weight=w)
            node_cnt += 1
            hyperedges += 1

    return (G, terms_df, boundary_weight, hyperedges)

def graph_vis_build_tanner_graph_from_dem(dem):
    return dem_graph.build_tanner_graph_from_dem(dem, extract_dem_info=extract_dem_info)

event_to_active_ids = dem_graph.event_to_active_ids


## Visualize using `matplotlib.animation`

In [ ]:
from matplotlib import animation
from IPython.display import HTML

def visualize_det_on_graph(
    G: nx.Graph,
    Gtype: Literal['matching', 'tanner'],
    det_events: str,
):
    active = event_to_active_ids(det_events)
    residual = set(G.nodes) - set(active)
    if Gtype == 'matching':
        residual -= set(G.graph['virtual_boundary_nodes'])
    pos = extract_coord_from_graph(G, dim=3)
    active_nodes = np.array([pos[d] for d in active])

    if Gtype == 'matching':
        virtual_boundary_nodes = np.array([pos[d] for d in G.graph['virtual_boundary_nodes']])
    residual_nodes = np.array([pos[d] for d in residual])

    if Gtype == 'tanner':
        residual_check = np.array([pos[d] for d in residual if G.nodes[d].get('kind') == 'detector'])
        residual_variable = np.array([pos[d] for d in residual if G.nodes[d].get('kind') == 'error'])
    ae = [(u, v) for u, v in G.edges if u in active or v in active]
    affected_edges = np.array([(pos[u], pos[v]) for u, v in ae])
    residual_edges = np.array([(pos[u], pos[v]) for u, v in G.edges - ae])

    def _init():
        ax.clear()
        if Gtype == 'matching':
            ax.scatter(*active_nodes.T, alpha=0.2, s=100, color='red')
            ax.scatter(*virtual_boundary_nodes.T, alpha=0.2, s=100, color='green')
            ax.scatter(*residual_nodes.T, alpha=0.2, s=100, color='blue')
        else:
            ax.scatter(
                *active_nodes.T,
                alpha=0.2,
                s=100,
                color='red',
                marker='s',
            )
            ax.scatter(
                *residual_check.T,
                alpha=0.2,
                s=100,
                color='blue',
                marker='s',
            )
            ax.scatter(
                *residual_variable.T,
                alpha=0.2,
                s=100,
                color='blue',
                marker='o',
            )

        for vizedge in affected_edges:
            ax.plot(*vizedge.T, color='red')
        for vizedge in residual_edges:
            ax.plot(*vizedge.T, color='gray')
        ax.grid(False)
        ax.set_axis_off()

    def _frame_update(index):
        ax.view_init(index * 0.2, index * 0.5)

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    fig.tight_layout()
    ani = animation.FuncAnimation(
        fig,
        _frame_update,
        init_func=_init,
        interval=50,
        cache_frame_data=False,
        frames=100,
    )

    return ani


### BPOSD fails but Tesseract succeeds

In [ ]:
noise_model = 'phenomenological'
graph_vis_data_dir = graph_vis_data_paths.output('cases', noise_model=noise_model)
PLOT_DIR = graph_vis_data_paths.output('graph_vis')
PLOT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
bfts_cases_path = graph_vis_data_paths.output('ca_bfts_cases_csv_1', noise_model=noise_model)
bfts_cases = pd.read_csv(bfts_cases_path, dtype=object)

bfts_cases


In [ ]:
case = bfts_cases.iloc[-5]
graph_vis_code = graph_vis_ca_codes[case['code']]
p = float(case['p'])
graph_vis_builder = graph_vis_CircuitBuilder(graph_vis_code)
graph_vis_circ = graph_vis_builder.get_circuit(noise_model=noise_model, error_rate=p, d_rounds=3)
dem = graph_vis_circ.detector_error_model()
MG, graph_vis_, graph_vis_, graph_vis_ = build_matching_graph_from_dem(dem)
graph_vis_tg, graph_vis_ = graph_vis_build_tanner_graph_from_dem(dem)


In [ ]:
graph_vis_circ.diagram('timeline-svg')


In [ ]:
graph_vis_ani = visualize_det_on_graph(MG, Gtype='matching', det_events=case['detector_events'])
save_path = graph_vis_data_paths.output('mg_bfts_phenomenological_CA_9_2_5_p_p_det_parameter_2_gif', p=p, parameter_2=case['detector_events'])
graph_vis_ani.save(save_path, writer='pillow', fps=20)
HTML(graph_vis_ani.to_jshtml())


In [ ]:
graph_vis_ani = visualize_det_on_graph(graph_vis_tg, Gtype='tanner', det_events=case['detector_events'])
save_path = graph_vis_data_paths.output('tg_bfts_phenomenological_CA_9_2_5_p_p_det_parameter_2_gif', p=p, parameter_2=case['detector_events'])
graph_vis_ani.save(save_path, writer='pillow', fps=20)
HTML(graph_vis_ani.to_jshtml())


### BPOSD succeeds but Tesseract fails

In [ ]:
noise_model = 'phenomenological'
graph_vis_data_dir = graph_vis_data_paths.output('cases', noise_model=noise_model)
PLOT_DIR = graph_vis_data_paths.output('graph_vis')
PLOT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
bstf_cases_path = graph_vis_data_paths.output('ca_bstf_cases_csv_2', noise_model=noise_model)
bstf_cases = pd.read_csv(bstf_cases_path, dtype=object)

bstf_cases


In [ ]:
case = bstf_cases.iloc[-5]
graph_vis_code = graph_vis_ca_codes[case['code']]
p = float(case['p'])
graph_vis_builder = graph_vis_CircuitBuilder(graph_vis_code)
graph_vis_circ = graph_vis_builder.get_circuit(noise_model=noise_model, error_rate=p, d_rounds=3)
dem = graph_vis_circ.detector_error_model()
MG, graph_vis_, graph_vis_, graph_vis_ = build_matching_graph_from_dem(dem)
graph_vis_tg, graph_vis_ = graph_vis_build_tanner_graph_from_dem(dem)


In [ ]:
graph_vis_ani = visualize_det_on_graph(MG, Gtype='matching', det_events=case['detector_events'])
save_path = graph_vis_data_paths.output('mg_bstf_phenomenological_CA_9_2_5_p_p_det_parameter_2_gif', p=p, parameter_2=case['detector_events'])
graph_vis_ani.save(save_path, writer='pillow', fps=20)
HTML(graph_vis_ani.to_jshtml())


In [ ]:
graph_vis_ani = visualize_det_on_graph(graph_vis_tg, Gtype='tanner', det_events=case['detector_events'])
save_path = graph_vis_data_paths.output('tg_bstf_phenomenological_CA_9_2_5_p_p_det_parameter_2_gif', p=p, parameter_2=case['detector_events'])
graph_vis_ani.save(save_path, writer='pillow', fps=20)
HTML(graph_vis_ani.to_jshtml())


## Visualize using plotly

In [ ]:
import plotly
import plotly.graph_objects as go
import numpy as np
from utils.graph_vis import visualize_det_on_graph_plotly


### Load the four outcome groups

Read the circuit-level case files used by the interactive examples below. Keep BPOSD-failure/Tesseract-success, the reverse disagreement, both-success and both-failure cases separate.


In [ ]:
noise_model = 'circuit_level'
graph_vis_data_dir = graph_vis_data_paths.output('cases', noise_model=noise_model)
PLOT_DIR = graph_vis_data_paths.output('graph_vis')
PLOT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
bfts_cases_path = graph_vis_data_paths.output('ca_bfts_cases_csv_3', noise_model=noise_model)
bstf_cases_path = graph_vis_data_paths.output('ca_bstf_cases_csv_4', noise_model=noise_model)
bs_cases_path = graph_vis_data_paths.output('ca_bs_cases_csv_5', noise_model=noise_model)
bf_cases_path = graph_vis_data_paths.output('ca_bf_cases_csv_6', noise_model=noise_model)
graph_vis_types = {
    'code': str,
    'p': float,
    'detector_events': str,
    'label': str,
}

bfts_cases = pd.read_csv(bfts_cases_path, dtype=graph_vis_types)
bstf_cases = pd.read_csv(bstf_cases_path, dtype=graph_vis_types)
bs_cases = pd.read_csv(bs_cases_path, dtype=graph_vis_types)
bf_cases = pd.read_csv(bf_cases_path, dtype=graph_vis_types)

bfts_cases['label'] = '01'
bstf_cases['label'] = '10'
bs_cases['label'] = '11'
bf_cases['label'] = '00'
bfts_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
bstf_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
bs_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
bf_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)


### BPOSD fails but Tesseract succeeds

In [ ]:
bfts_cases


In [ ]:
case = bfts_cases.iloc[-5]
graph_vis_code = graph_vis_ca_codes[case['code']]
p = float(case['p'])
graph_vis_builder = graph_vis_CircuitBuilder(graph_vis_code)
graph_vis_circ = graph_vis_builder.get_circuit(noise_model=noise_model, error_rate=p, d_rounds=3)
dem = graph_vis_circ.detector_error_model()
MG, graph_vis_, graph_vis_, graph_vis_ = build_matching_graph_from_dem(dem)
graph_vis_tg, graph_vis_ = graph_vis_build_tanner_graph_from_dem(dem)


In [ ]:
graph_vis_fig = visualize_det_on_graph_plotly(MG, Gtype='matching', det_events=case['detector_events'])
graph_vis_fig.show()


In [ ]:
graph_vis_fig = visualize_det_on_graph_plotly(graph_vis_tg, Gtype='tanner', det_events=case['detector_events'])
graph_vis_fig.show()


### BPOSD succeeds but Tesseract fails

In [ ]:
bstf_cases


In [ ]:
case = bstf_cases.iloc[-5]
graph_vis_code = graph_vis_ca_codes[case['code']]
p = float(case['p'])
graph_vis_builder = graph_vis_CircuitBuilder(graph_vis_code)
graph_vis_circ = graph_vis_builder.get_circuit(noise_model=noise_model, error_rate=p, d_rounds=3)
dem = graph_vis_circ.detector_error_model()
MG, graph_vis_, graph_vis_, graph_vis_ = build_matching_graph_from_dem(dem)
graph_vis_tg, graph_vis_ = graph_vis_build_tanner_graph_from_dem(dem)


In [ ]:
graph_vis_fig = visualize_det_on_graph_plotly(MG, Gtype='matching', det_events=case['detector_events'])
graph_vis_fig.show()


In [ ]:
graph_vis_fig = visualize_det_on_graph_plotly(graph_vis_tg, Gtype='tanner', det_events=case['detector_events'])
graph_vis_fig.show()


### Both succeed

In [ ]:
bs_cases


In [ ]:
case = bs_cases.iloc[-5]
graph_vis_code = graph_vis_ca_codes[case['code']]
p = float(case['p'])
graph_vis_builder = graph_vis_CircuitBuilder(graph_vis_code)
graph_vis_circ = graph_vis_builder.get_circuit(noise_model=noise_model, error_rate=p, d_rounds=3)
dem = graph_vis_circ.detector_error_model()
MG, graph_vis_, graph_vis_, graph_vis_ = build_matching_graph_from_dem(dem)
graph_vis_tg, graph_vis_ = graph_vis_build_tanner_graph_from_dem(dem)


In [ ]:
graph_vis_fig = visualize_det_on_graph_plotly(MG, Gtype='matching', det_events=case['detector_events'])
graph_vis_fig.show()


In [ ]:
graph_vis_fig = visualize_det_on_graph_plotly(graph_vis_tg, Gtype='tanner', det_events=case['detector_events'])
graph_vis_fig.show()


### Both Fail

In [ ]:
bf_cases


In [ ]:
case = bf_cases.iloc[-5]
graph_vis_code = graph_vis_ca_codes[case['code']]
p = float(case['p'])
graph_vis_builder = graph_vis_CircuitBuilder(graph_vis_code)
graph_vis_circ = graph_vis_builder.get_circuit(noise_model=noise_model, error_rate=p, d_rounds=3)
dem = graph_vis_circ.detector_error_model()


In [ ]:
graph_vis_fig = visualize_det_on_graph_plotly(MG, Gtype='matching', det_events=case['detector_events'])
graph_vis_fig.show()


In [ ]:
graph_vis_fig = visualize_det_on_graph_plotly(graph_vis_tg, Gtype='tanner', det_events=case['detector_events'])
graph_vis_fig.show()


# Check

### Inspect a structural-feature example

Select the named WL-hash group from the structural table and display a representative detector pattern. A shared hash is a grouping aid, not proof of an identical decoding mechanism.


In [ ]:
graph_vis_data_dir = graph_vis_data_paths.input('phenomenological_trapping_structure')
graph_vis_analysis_df = pd.read_csv(graph_vis_data_paths.input('event_graph_features_csv_7'), dtype=object)


In [ ]:
target_case = graph_vis_analysis_df[graph_vis_analysis_df['WL_hash'] == '509b0c29ecac32856bba94e0b68e0967']
target_case


In [ ]:
case = target_case.iloc[0]
graph_vis_code = graph_vis_ca_codes[case['code']]
p = float(case['p'])
graph_vis_builder = graph_vis_CircuitBuilder(graph_vis_code)
graph_vis_circ = graph_vis_builder.get_circuit(noise_model='phenomenological', error_rate=p, d_rounds=3)
dem = graph_vis_circ.detector_error_model()
MG, graph_vis_, graph_vis_, graph_vis_ = build_matching_graph_from_dem(dem)
graph_vis_tg, graph_vis_ = graph_vis_build_tanner_graph_from_dem(dem)


In [ ]:
graph_vis_fig = visualize_det_on_graph_plotly(graph_vis_tg, Gtype='tanner', det_events=case['detector_events'])
graph_vis_fig.show()


In [ ]:
visualize_det_on_graph_plotly(graph_vis_tg, Gtype='tanner', det_events='000010011000').show()
